# 🚀 PaliGemma Math Recognition Training

**Optimized for A100 GPUs with checkpoint recovery**

**Features:**
- ✅ Auto-resume from checkpoints (survives tab closure)
- ✅ Optimized batch sizes & data loading
- ✅ Mixed precision training
- ✅ Frequent checkpointing to Google Drive

**Setup:** HuggingFace token in Colab secrets (🔑) as `HF_TOKEN`

In [ ]:
# ============================================================================
# SETUP: Environment, Dependencies & Authentication
# ============================================================================

import os
import torch
import subprocess
import glob
import time
import urllib.request
from google.colab import drive, userdata
from huggingface_hub import login

# Mount Drive
drive.mount('/content/drive', force_remount=False)

# Setup paths
WORK_DIR = '/content/drive/MyDrive/math-training'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir('/content')

# GPU detection
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Auto batch size
if 'H100' in gpu_name:
    BATCH_SIZE, NUM_WORKERS = 20, 8
elif 'A100' in gpu_name:
    BATCH_SIZE, NUM_WORKERS = 18, 8
else:
    BATCH_SIZE, NUM_WORKERS = 8, 4

print(f"\n🖥️  GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")
print(f"⚙️  Batch: {BATCH_SIZE} | Workers: {NUM_WORKERS}")

# Install dependencies (fix Pillow version issue)
%pip install -q --upgrade \
    torch>=2.5.0 transformers>=4.45.0 peft>=0.11.0 \
    huggingface_hub>=0.23.0 accelerate>=0.30.0 datasets>=3.0.0 \
    sentencepiece>=0.2.0 "protobuf>=3.20.0,<4.0.0" bitsandbytes>=0.43.0 \
    timm>=1.0.0 "pillow>=10.3.0" "numpy>=1.26.2,<1.27.0" tqdm>=4.66.0

# HuggingFace auth
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print("✅ HuggingFace authenticated")
except Exception as e:
    print(f"❌ Auth failed: {e}\nSetup: https://huggingface.co/settings/tokens")
    raise

print("✅ Setup complete")

In [ ]:
# ============================================================================
# DATASET & PROJECT FILES
# ============================================================================

DATASET_URL = "https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz"
LOCAL_DATA_DIR = "/content/mathwriting-2024"
DRIVE_TARBALL = f"{WORK_DIR}/mathwriting-2024.tgz"

# Check if dataset exists
if os.path.exists(f"{LOCAL_DATA_DIR}/train"):
    train_count = len(glob.glob(f"{LOCAL_DATA_DIR}/train/*.inkml"))
    if train_count > 100000:
        print("✅ Dataset already extracted")
        DATA_DIR = LOCAL_DATA_DIR
    else:
        raise RuntimeError("Incomplete dataset")
else:
    # Find or download tarball
    tarball_path = None
    for path in [DRIVE_TARBALL, f"{WORK_DIR}/mathwriting-2024.tgz", 
                 "/content/drive/MyDrive/mathwriting-2024.tgz"]:
        if os.path.exists(path):
            tarball_path = path
            break
    
    if not tarball_path:
        print("📥 Downloading dataset...")
        !wget -q --show-progress {DATASET_URL} -O {DRIVE_TARBALL}
        tarball_path = DRIVE_TARBALL
    
    print("📦 Extracting...")
    result = subprocess.run(["tar", "-xzf", tarball_path, "-C", "/content/"],
                           capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Extraction failed: {result.stderr}")
    DATA_DIR = LOCAL_DATA_DIR

# Download project files if needed
if not os.path.exists('data_preprocessing.py'):
    print("📥 Downloading project files...")
    files = [
        ('data_preprocessing.py', 'https://raw.githubusercontent.com/hudsonmp/realtime-math/main/data_preprocessing.py'),
        ('train.py', 'https://raw.githubusercontent.com/hudsonmp/realtime-math/main/train.py'),
    ]
    for filename, url in files:
        try:
            urllib.request.urlretrieve(url, filename)
            print(f"   ✅ {filename}")
        except Exception as e:
            print(f"   ❌ {filename}: {e}")

assert os.path.exists('data_preprocessing.py'), "Missing data_preprocessing.py"
print(f"✅ Dataset ready: {DATA_DIR}")

In [ ]:
# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================

import json
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration, get_scheduler
from peft import LoraConfig, get_peft_model, PeftModel
from data_preprocessing import MathWritingDataset, LaTeXTokenizer
from tqdm import tqdm

# Hyperparameters
EPOCHS = 10
GRAD_ACCUM = 2
LEARNING_RATE = 2e-4
WARMUP_STEPS = 500
MAX_GRAD_NORM = 1.0
VALIDATION_INTERVAL = 2
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05

CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
STATE_FILE = f"{CHECKPOINT_DIR}/training_state.json"
device = "cuda"

# Resume from checkpoint?
RESUME_FROM_CHECKPOINT = True  # Set False to start fresh

def save_training_state(epoch, best_cer, global_step):
    """Save training state for recovery."""
    state = {
        'epoch': epoch,
        'best_cer': best_cer,
        'global_step': global_step
    }
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f)

def load_training_state():
    """Load training state if exists."""
    if RESUME_FROM_CHECKPOINT and os.path.exists(STATE_FILE):
        with open(STATE_FILE, 'r') as f:
            return json.load(f)
    return {'epoch': 0, 'best_cer': float('inf'), 'global_step': 0}

print("="*70)
print("🚀 TRAINING CONFIGURATION")
print("="*70)
print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")
print(f"Epochs: {EPOCHS} | Batch: {BATCH_SIZE} (eff: {BATCH_SIZE * GRAD_ACCUM})")
print(f"LoRA: r={LORA_R}, α={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Resume: {RESUME_FROM_CHECKPOINT}")
print("="*70)

In [ ]:
# ============================================================================
# LOAD MODEL & DATASETS
# ============================================================================

print("\n📦 Loading PaliGemma-3B...")
processor = AutoProcessor.from_pretrained("google/paligemma-3b-pt-224")

# Check for existing checkpoint
state = load_training_state()
start_epoch = state['epoch'] if RESUME_FROM_CHECKPOINT else 0
best_cer = state['best_cer']

if RESUME_FROM_CHECKPOINT and start_epoch > 0:
    print(f"🔄 Resuming from epoch {start_epoch}...")
    base_model = PaliGemmaForConditionalGeneration.from_pretrained(
        "google/paligemma-3b-pt-224",
        torch_dtype=torch.bfloat16,
        device_map=None
    )
    checkpoint_path = f"{CHECKPOINT_DIR}/epoch_{start_epoch}"
    if os.path.exists(checkpoint_path):
        model = PeftModel.from_pretrained(base_model, checkpoint_path)
        print(f"✅ Loaded checkpoint: {checkpoint_path}")
    else:
        print("⚠️  Checkpoint not found, starting fresh")
        model = PaliGemmaForConditionalGeneration.from_pretrained(
            "google/paligemma-3b-pt-224",
            torch_dtype=torch.bfloat16,
            device_map=None
        )
        lora_config = LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
            bias="none", task_type="CAUSAL_LM"
        )
        model = get_peft_model(model, lora_config)
        start_epoch = 0
else:
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        "google/paligemma-3b-pt-224",
        torch_dtype=torch.bfloat16,
        device_map=None
    )
    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        bias="none", task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)

model.to(device)

# Compile for speedup
if hasattr(torch, 'compile'):
    print("⚡ Compiling model...")
    model = torch.compile(model, mode='reduce-overhead')

model.print_trainable_parameters()

# Datasets
print("\n📊 Loading datasets...")
train_ds = MathWritingDataset(DATA_DIR, split='train')
valid_ds = MathWritingDataset(DATA_DIR, split='valid')
print(f"   Train: {len(train_ds):,} | Valid: {len(valid_ds):,}")

latex_tokenizer = LaTeXTokenizer()

# Collate function
def collate_fn(batch):
    stroke_texts = [item['stroke_text'] for item in batch]
    images = [item['image'] for item in batch]
    labels = [item['label'] for item in batch]
    
    inputs = processor(text=stroke_texts, images=images, padding="longest",
                     truncation=True, max_length=1024, return_tensors="pt")
    label_encodings = processor.tokenizer(labels, padding="max_length",
                                         truncation=True, max_length=64, return_tensors="pt")
    
    batch_size, seq_length = inputs['input_ids'].shape
    labels_tensor = torch.full((batch_size, seq_length), -100, dtype=torch.long)
    for i, label_ids in enumerate(label_encodings['input_ids']):
        label_length = (label_ids != processor.tokenizer.pad_token_id).sum().item()
        labels_tensor[i, -label_length:] = label_ids[:label_length]
    
    inputs['labels'] = labels_tensor
    return inputs

# Data loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS,
                          pin_memory=True, persistent_workers=True if NUM_WORKERS > 0 else False,
                          prefetch_factor=2 if NUM_WORKERS > 0 else None)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=NUM_WORKERS,
                         pin_memory=True, persistent_workers=True if NUM_WORKERS > 0 else False)

# Optimizer & scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
num_training_steps = EPOCHS * len(train_loader) // GRAD_ACCUM
scheduler = get_scheduler("cosine", optimizer=optimizer,
                         num_warmup_steps=WARMUP_STEPS, num_training_steps=num_training_steps)
scaler = GradScaler()

print(f"📈 Steps/epoch: {len(train_loader)} | Total steps: {num_training_steps}")

In [ ]:
# ============================================================================
# TRAINING LOOP (with checkpoint recovery)
# ============================================================================

print("\n" + "="*70)
print("🚀 STARTING TRAINING")
print("="*70)

model.train()
global_step = state.get('global_step', 0)
start_time = time.time()

for epoch in range(start_epoch, EPOCHS):
    print(f"\n{'='*70}")
    print(f"📅 EPOCH {epoch + 1}/{EPOCHS}")
    print('='*70)
    
    epoch_loss = 0
    optimizer.zero_grad()
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    
    for step, batch in enumerate(progress_bar):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        
        with autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM
        
        scaler.scale(loss).backward()
        
        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
        
        epoch_loss += loss.item() * GRAD_ACCUM
        
        if step % 100 == 0:
            progress_bar.set_postfix({
                'loss': f'{loss.item() * GRAD_ACCUM:.4f}',
                'lr': f'{scheduler.get_last_lr()[0]:.2e}'
            })
        
        # Save state every 500 steps (survives tab closure)
        if global_step % 500 == 0:
            save_training_state(epoch, best_cer, global_step)
    
    avg_loss = epoch_loss / len(train_loader)
    elapsed = (time.time() - start_time) / 3600
    
    print(f"\n📊 Epoch {epoch+1} Results:")
    print(f"   Train Loss: {avg_loss:.4f} | Time: {elapsed:.2f}h")
    
    # Validation
    if (epoch + 1) % VALIDATION_INTERVAL == 0 or epoch == EPOCHS - 1:
        print("\n🔍 Running validation...")
        model.eval()
        val_loss = total_cer = num_samples = 0
        
        with torch.no_grad():
            for batch in tqdm(valid_loader, desc="Validation"):
                inputs = {k: v.to(device, non_blocking=True) 
                         for k, v in batch.items() if k != 'labels'}
                labels = batch['labels'].to(device, non_blocking=True)
                
                with autocast(dtype=torch.bfloat16):
                    outputs = model(**{**inputs, 'labels': labels})
                    val_loss += outputs.loss.item()
                    generated = model.generate(**inputs, max_length=64, 
                                             do_sample=False, num_beams=1)
                
                for pred_ids, label_ids in zip(generated, labels):
                    pred_text = processor.decode(pred_ids, skip_special_tokens=True)
                    label_text = processor.decode(label_ids[label_ids != -100], 
                                                skip_special_tokens=True)
                    cer = latex_tokenizer.compute_cer(pred_text, label_text)
                    total_cer += cer
                    num_samples += 1
        
        avg_val_loss = val_loss / len(valid_loader)
        avg_cer = total_cer / num_samples if num_samples > 0 else 0
        
        print(f"\n📊 Validation: Loss={avg_val_loss:.4f} | CER={avg_cer:.4f}")
        model.train()
        
        # Save best model
        if avg_cer < best_cer:
            print(f"🎉 New best CER: {best_cer:.4f} → {avg_cer:.4f}")
            best_cer = avg_cer
            save_path = f"{CHECKPOINT_DIR}/best_model"
            model.save_pretrained(save_path)
            processor.save_pretrained(save_path)
            print(f"✅ Best model saved")
    
    # Checkpoint every epoch (for recovery)
    save_path = f"{CHECKPOINT_DIR}/epoch_{epoch+1}"
    model.save_pretrained(save_path)
    processor.save_pretrained(save_path)
    save_training_state(epoch + 1, best_cer, global_step)
    print(f"💾 Checkpoint saved: epoch_{epoch+1}")

# Final save
final_path = f"{CHECKPOINT_DIR}/final_model"
model.save_pretrained(final_path)
processor.save_pretrained(final_path)

total_time = (time.time() - start_time) / 3600
print("\n" + "="*70)
print("🎉 TRAINING COMPLETE!")
print("="*70)
print(f"Total time: {total_time:.2f}h | Best CER: {best_cer:.4f}")
print(f"Models: {CHECKPOINT_DIR}")
print("="*70)

In [ ]:
# ============================================================================
# OPTIONAL: Test Inference
# ============================================================================

from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
from data_preprocessing import MathWritingDataset

print("Loading best model for inference...")

base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma-3b-pt-224",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, f"{CHECKPOINT_DIR}/best_model")
processor = AutoProcessor.from_pretrained(f"{CHECKPOINT_DIR}/best_model")

print("✅ Model loaded!")

# Test on a sample
test_ds = MathWritingDataset(DATA_DIR, split='test')
sample = test_ds[0]

inputs = processor(
    text=sample['stroke_text'],
    images=sample['image'],
    return_tensors="pt"
).to("cuda")

generated = model.generate(**inputs, max_length=64)
prediction = processor.decode(generated[0], skip_special_tokens=True)

print(f"\n🧪 Sample Test:")
print(f"   Ground Truth: {sample['label']}")
print(f"   Prediction:   {prediction}")

---

## 📝 Notes

**Checkpoint Recovery:**
- Training state saved every 500 steps & every epoch
- Set `RESUME_FROM_CHECKPOINT = True` to auto-resume
- Checkpoints saved to Google Drive (survives tab closure)

**Performance:**
- A100: ~8-12 hours | H100: ~6-10 hours
- Checkpoints every epoch + best model saved
- Mixed precision + torch.compile() for speed

**Tab Closure:**
- ⚠️ Colab stops when tab closes (Colab limitation)
- ✅ Checkpoints saved to Drive - manually resume by running cells again
- Set `RESUME_FROM_CHECKPOINT = True` to continue from last checkpoint